In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
print(os.listdir('/content/drive/MyDrive/'))

['Getting started.pdf', 'ADMISSION LETTER_page-0001.jpg', '210280107044_Sign.jpg', 'Classroom', 'IMG_20230820_211628.jpg', 'Colab Notebooks', 'Blockchain Technology & Career in Blockchain.gdoc', '12th Marksheet.jpg', 'Pancard.jpg', 'Jeet Application Documents', 'Doc Scanner', 'Doc Scanner Upload', 'Jeet_SOP.docx', 'Jeet Resume (Professional).docx', 'Jeet Resume.docx', 'Jeet_Resume.pdf', 'JN TATA SOP.gdoc', 'Jeet_University at Buffalo', 'KC Mahindra SOP.gdoc', 'Rare Visa Questions.gdoc', 'Passport_Photo.jpg', 'F1 VISA INTERVIEW QUESTIONS.gdoc', 'Technical Questions.gdoc', 'Practical Assessment Inference.gdoc', 'Cover Letter Template.gdoc', 'Forensic Detective Report.gdoc', 'Forensic_Detective_Report[1].gdoc', 'columbia1_label_1Z309A259010096713_copy.pdf', 'Regression Report.gdoc', 'Analysis and Reflection Report.gdoc', 'Classification_Report.gdoc', 'Cover Letter.gdoc', 'CV_GSA_Editor.gdoc', 'Saved from the Google app', 'Untitled document (2).gdoc', 'CV_GA.gdoc', 'Colab_CV.gdoc', 'Colab.

In [ ]:
import sys
folder_path = '/content/drive/MyDrive/assignment_2_part_1'

if folder_path not in sys.path:
    sys.path.append(folder_path)

In [4]:
# Imports
from environment import WumpusWorldEnvironment

# Environment

We will be working with an implementation of the Wumpus World environment. The environment comes from the book "Artificial Intelligence: A Modern Approach" by Stuart J. Russell and Peter Norvig. 

### ENVIRONMENT DETAILS:

The environment is a 6 x 6 grid world containing a total of 36 grid blocks. 

#### ENVIRONMENT OBJECTS:
The environment consists of the following objects:

1. **Agent** - The agent starts in the grid block at the bottom left corner whose co-ordinates are [0, 0]. The goal of our agent is to collect the Gold while avoiding the Wumpus and the pits. 

2. **Wumpus** - The monster which would eat the agent if they are in the same grid block.

3. **Pit** - The agent must avoid falling into the pits. 

4. **Gold** - The agent must collect the Gold.

5. **Breeze** - Breeze surrounds the Pits and warn the agent of a Pit in an adjacent grid block.

6. **Stench** - Stench surrounds the Wumpus and warns the agent of the Wumpus in an adjacent grid block.

#### ENVIRONMENT OBSERVATIONS:

Our implementation of the environment provides you with four different types of observations:

1. **Integer** - Integer in the range [0 - 35]. This represents the grid block the agent is in. E.g., if the agent is in the bottom left grid block (starting position) the observation would be 0, if the agent is in the grid block containing the Gold the observation would be 34, if the agent is in the top right grid block the observation would be 35.

2. **Vector** - 

    **2.1.** A vector of length 2 representing the agent co-ordinates. The first entry represents the x co-ordinate and the second entry represets the y co-ordinate. E.g., if the agent is in the bottom left grid block (starting position) the observation would be [0, 0], if the agent is in the grid block containing the Gold the observation would be [4, 5], if the agent is in the top right grid block the observation would be [5, 5].
    
    **2.2.** A vector of length 36 representing the one-hot encoding of the integer observation (refer type 1 above). E.g., if the agent is in the bottom left grid block (starting position) the observation would be [1, 0, ..., 0, 0], if the agent is in the grid block containing the Gold the observation would be [0, 0, ..., 1, 0], if the agent is in the top right grid block the observation would be [0, 0, ..., 0, 1].


3. **Image** - Image render of the environment returned as an NumPy array. The image size is 84 * 84 (same size used in the DQN paper). E.g., if the agent is in the bottom right grid block the observation is:

    Observation: (84 * 84)

     [[255 255 255 ... 255 255 255]

     [255 255 255 ... 255 255 255]

     [255 255 255 ... 255 255 255]

     ...

     [255 255 255 ... 255 255 255]

     [255 255 255 ... 255 255 255]

     [255 255 255 ... 255 255 255]]

    Observation type: <class 'numpy.ndarray'>

    Observation Shape: (84, 84)

    Visually, it looks like:
    <img src="./images/environment_render.png" width="500" height="500">
    

4. **Float** - Float in the range [0 - $\infty$] representing the time elapsed in seconds. 

#### ENVIRONMENT ACTIONS:

Our implementation of the environment provides you with three different types of actions:

1. **Discrete** - Integer in the range [0 - 3] representing the four actions possible in the environment as follows: 0 - Right 1 - Left 2 - Up 3 - Down.

2. **Multi-Discrete** - Array of length four where each element takes binary values 0 or 1. Array elements represent if we take a particular action. Array element with index 0 corresponds to the right action, index 1 corresponds to the left action, index 2 corresponds to the up action, and index 3 corresponds to the down action. E.g., 
   action = [1, 0, 0, 0] would result in the agent moving right.
   action = [1, 0, 1, 0] would result in the agent moving right and up.
   action = [0, 1, 0, 1] would result in the agent moving left and down.

3. **Continuous** - Float in the range [-1, 1] determining whether the agent will go left, right, up, or down as follows:

    if -1 <= action <= -0.5:
        Go Right.
    elif -0.5 < action <= 0:
        Go Left.
    elif 0 < action <= 0.5:
        Go Up.
    elif 0.5 < action <= 1:
        Go Down.
        
### TASK IS TO USE A NEURAL NETWORK TO WORK WITH ALL FOUR TYPES OF OBSERVATIONS AND ALL THREE TYPES OF  ACTIONS.

<img src="./images/wumpus_world_environment.jpg" width="600" height="600">

## Observation Type - Integer, Action Type - Discrete

created a sequential dense neural network with 1 hidden layer having 64 neurons and the output layer having 4 neurons. The input to the neural network is an integer. The output of the neural network is an array represeting the Q-values from which you will choose an action.

The following figure shows the network structure you will have to use:

<img src="./images/neural_network_structures/neural_network_1_64_4.png">

In [ ]:
environment = WumpusWorldEnvironment(observation_type='integer', action_type='discrete')
observation, info = environment.reset()

import torch
import torch.nn as nn

class QNet1(nn.Module):
    def __init__(self):
        super(QNet1, self).__init__()
        self.fc1 = nn.Linear(1, 64)
        self.fc2 = nn.Linear(64, 4)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = QNet1()
obs_tensor = torch.tensor([observation], dtype=torch.float32)
q_values = model(obs_tensor)

print("Observation:", observation)
print("Q-values:", q_values.detach().numpy())

Observation: 0
Q-values: [ 0.03363022  0.02255484 -0.16688095  0.15773194]


## Observation Type - Vector (2.1), Action Type - Discrete
created a sequential dense neural network with 1 hidden layer having 64 neurons and the output layer having 4 neurons. The input to the neural network is a vector of length 2. The output of the neural network is an array represeting the Q-values from which you will choose an action.

The following figure shows the network structure you will have to use:

<img src="./images/neural_network_structures/neural_network_2_64_4.png">

In [ ]:
environment = WumpusWorldEnvironment(observation_type='vector', action_type='discrete')
observation, info = environment.reset()

class QNet2(nn.Module):
    def __init__(self):
        super(QNet2, self).__init__()
        self.fc1 = nn.Linear(2, 64)
        self.fc2 = nn.Linear(64, 4)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = QNet2()
obs_tensor = torch.tensor([observation], dtype=torch.float32)
q_values = model(obs_tensor)

print("Observation:", observation)
print("Q-values:", q_values.detach().numpy())


Observation: [0 0]
Q-values: [[ 0.02326486 -0.07223234  0.00387963 -0.32090306]]


/tmp/ipython-input-1050/911066400.py:19: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  obs_tensor = torch.tensor([observation], dtype=torch.float32)


## Observation Type - Vector (2.2), Action Type - Discrete


The following figure shows the network structure you will have to use:

<img src="./images/neural_network_structures/neural_network_36_64_4.png">

In [ ]:
environment = WumpusWorldEnvironment(observation_type='integer', action_type='discrete')
observation, info = environment.reset()

import numpy as np

class QNet3(nn.Module):
    def __init__(self):
        super(QNet3, self).__init__()
        self.fc1 = nn.Linear(36, 64)
        self.fc2 = nn.Linear(64, 4)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)
    
model = QNet3()

one_hot_obs = np.zeros(36)
one_hot_obs[observation] = 1
obs_tensor = torch.tensor([one_hot_obs], dtype=torch.float32)

q_values = model(obs_tensor)

print("Observation:", observation)
print("One-hot Observation:", one_hot_obs)
print("Q-values:", q_values.detach().numpy())


Observation: 0
One-hot Observation: [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Q-values: [[-0.16265821 -0.05174369 -0.05464371  0.01441452]]


## Observation Type - Image, Action Type - Discrete

The following figure shows the network structure you will have to use:

<img src="./images/neural_network_structures/convolutional_neural_network_84x84_128_64_4.png">

In [ ]:
environment = WumpusWorldEnvironment(observation_type='image', action_type='discrete')
observation, info = environment.reset()

class Cnn(nn.Module):
    def __init__(self):
        super(Cnn, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels = 128, kernel_size=3)
        self.fc1 = nn.Linear(128 * 82 * 82, 64) 
        self.fc2 = nn.Linear(64, 4)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = x.view(x.size(0), -1)  # Flatten the output of cnn
        x = torch.relu(self.fc1(x))
        return self.fc2(x)
    
model = Cnn()
obs_tensor = torch.tensor(observation, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  
q_values = model(obs_tensor)

print("Observation shape:", observation.shape)
print("Q-values:", q_values.detach().numpy())

Observation shape: (84, 84)
Q-values: [[-11.37049   -12.43842    -6.9549255  10.033916 ]]


## Observation Type - Float, Action Type - Discrete

The following figure shows the network structure you will have to use:

<img src="./images/neural_network_structures/neural_network_1_64_4.png">

In [ ]:
environment = WumpusWorldEnvironment(observation_type='float', action_type='discrete')
observation, info = environment.reset()

class FloatQNet(nn.Module):
    def __init__(self):
        super(FloatQNet, self).__init__()
        self.fc1 = nn.Linear(1, 64)
        self.fc2 = nn.Linear(64, 4)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = FloatQNet()
obs_tensor = torch.tensor([observation], dtype=torch.float32)
q_values = model(obs_tensor) 

print("Observation:", observation)
print("Q-values:", q_values.detach().numpy())

Observation: 5.91278076171875e-05
Q-values: [ 0.33352786 -0.04493092 -0.07546815 -0.17425554]


## Observation Type - Vector (2.2), Action Type - Multi-Discrete

The following figure shows the network structure you will have to use:

<img src="./images/neural_network_structures/neural_network_36_64_4_sigmoid.png">

In [ ]:
environment = WumpusWorldEnvironment(observation_type='integer', action_type='multi_discrete')
observation, info = environment.reset()

class MultiDiscreteQNet(nn.Module):
    def __init__(self):
        super(MultiDiscreteQNet, self).__init__()
        self.fc1 = nn.Linear(36, 64)
        self.fc2 = nn.Linear(64, 4)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.sigmoid(self.fc2(x))
    
model = MultiDiscreteQNet()
one_hot_obs = np.zeros(36)
one_hot_obs[observation] = 1.0
obs_tensor = torch.tensor([one_hot_obs], dtype=torch.float32)
action_probs = model(obs_tensor)

print("Observation:", observation)
print("Action probabilities:", action_probs.detach().numpy())
predicted_action = (action_probs.detach().numpy() >= 0.5).astype(int)
print("Predicted action:", predicted_action)

Observation: 0
Action probabilities: [[0.4867554  0.49857807 0.5009689  0.4889167 ]]
Predicted action: [[0 0 1 0]]


## Observation Type - Vector (2.2), Action Type - Continuous


The following figure shows the network structure you will have to use:

<img src="./images/neural_network_structures/neural_network_36_64_1.png">

In [ ]:
environment = WumpusWorldEnvironment(observation_type='integer', action_type='continuous')
observation, info = environment.reset()

class ContinuousQNet(nn.Module):
    def __init__(self):
        super(ContinuousQNet, self).__init__()
        self.fc1 = nn.Linear(36, 64)
        self.fc2 = nn.Linear(64, 1)
        self.tanh = nn.Tanh()

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.tanh(self.fc2(x))

model = ContinuousQNet()

one_hot_obs = np.zeros(36)
one_hot_obs[observation] = 1.0
obs_tensor = torch.tensor([one_hot_obs], dtype=torch.float32)
predicted_action = model(obs_tensor)

print("Observation:", observation)
print("Predicted action:", predicted_action.detach().numpy())

Observation: 0
Predicted action: [[0.04085875]]
